# Examine output

use this notebook to see how effective the processing is for 2022.

In [6]:
import requests

vars2021 = requests.get("https://api.census.gov/data/2021/acs/acs5/variables.json").json()["variables"]
vars2022 = requests.get("https://api.census.gov/data/2022/acs/acs5/variables.json").json()["variables"]

In [7]:
missing = newly_missing_in_2022

still_exist = []
gone = []

for var in missing:
    if var in vars2022:
        still_exist.append(var)
    else:
        gone.append(var)

print("Still exist in 2022:", len(still_exist))
print("Gone in 2022:", len(gone))

Still exist in 2022: 89
Gone in 2022: 0


So like, the variables still exist per the metadata (variables json provided by the ACS), but they are not present where I expect them (tiger product). First, let's ID all the variables by their census identifiers:

In [12]:
for var in still_exist:
    print(var, "->", vars2022[var]["label"])

B01001_003E -> Estimate!!Total:!!Male:!!Under 5 years
B01001_004E -> Estimate!!Total:!!Male:!!5 to 9 years
B01001_005E -> Estimate!!Total:!!Male:!!10 to 14 years
B01001_006E -> Estimate!!Total:!!Male:!!15 to 17 years
B01001_018E -> Estimate!!Total:!!Male:!!60 and 61 years
B01001_019E -> Estimate!!Total:!!Male:!!62 to 64 years
B01001_020E -> Estimate!!Total:!!Male:!!65 and 66 years
B01001_021E -> Estimate!!Total:!!Male:!!67 to 69 years
B01001_022E -> Estimate!!Total:!!Male:!!70 to 74 years
B01001_023E -> Estimate!!Total:!!Male:!!75 to 79 years
B01001_024E -> Estimate!!Total:!!Male:!!80 to 84 years
B01001_025E -> Estimate!!Total:!!Male:!!85 years and over
B01001_027E -> Estimate!!Total:!!Female:!!Under 5 years
B01001_028E -> Estimate!!Total:!!Female:!!5 to 9 years
B01001_029E -> Estimate!!Total:!!Female:!!10 to 14 years
B01001_030E -> Estimate!!Total:!!Female:!!15 to 17 years
B01001_042E -> Estimate!!Total:!!Female:!!60 and 61 years
B01001_043E -> Estimate!!Total:!!Female:!!62 to 64 year

In [13]:
import re
from collections import defaultdict

In [14]:
def variable_to_table_group(var: str) -> str | None:
    """
    Convert an ACS variable like B01003_001E to its table/group name B01003.
    """
    m = re.match(r"^([A-Z0-9]+)_\d+[A-Z]$", var)
    if m:
        return m.group(1)
    return None


def group_variables_by_table(vars_list: list[str]) -> dict[str, list[str]]:
    groups = defaultdict(list)
    unparsed = []

    for var in sorted(set(vars_list)):
        group = variable_to_table_group(var)
        if group is None:
            unparsed.append(var)
        else:
            groups[group].append(var)

    if unparsed:
        print("Could not parse these variables:")
        for var in unparsed:
            print("  ", var)

    return dict(sorted(groups.items()))


groups = group_variables_by_table(newly_missing_in_2022)

print(f"Unique table groups: {len(groups)}")
for group, vars_ in groups.items():
    print(f"{group}: {len(vars_)} vars")

Unique table groups: 17
B01001: 24 vars
B01003: 1 vars
B02001: 1 vars
B03002: 6 vars
B12001: 8 vars
B15002: 25 vars
B17010: 4 vars
B19001: 1 vars
B19013: 1 vars
B19301: 1 vars
B21001: 1 vars
B25002: 3 vars
B25003: 3 vars
B25024: 7 vars
B25058: 1 vars
B25077: 1 vars
C24010: 1 vars


## Another way to inspect the tables/groups

In [ ]:
def describe_group(group: str, vars_meta: dict) -> pd.DataFrame:
    rows = []
    for var, meta in vars_meta.items():
        if var.startswith(f"{group}_"):
            rows.append(
                {
                    "variable": var,
                    "label": meta.get("label"),
                    "concept": meta.get("concept"),
                    "predicateType": meta.get("predicateType"),
                    "group": meta.get("group"),
                }
            )
    return pd.DataFrame(rows).sort_values("variable")

In [16]:
# Example:
describe_group("B01001", vars2022).head(20)

,variable,label,concept,predicateType,group
40,B01001_001E,Estimate!!Total:,Sex by Age,int,B01001
39,B01001_002E,Estimate!!Total:!!Male:,Sex by Age,int,B01001
42,B01001_003E,Estimate!!Total:!!Male:!!Under 5 years,Sex by Age,int,B01001
41,B01001_004E,Estimate!!Total:!!Male:!!5 to 9 years,Sex by Age,int,B01001
45,B01001_005E,Estimate!!Total:!!Male:!!10 to 14 years,Sex by Age,int,B01001
43,B01001_006E,Estimate!!Total:!!Male:!!15 to 17 years,Sex by Age,int,B01001
44,B01001_007E,Estimate!!Total:!!Male:!!18 and 19 years,Sex by Age,int,B01001
47,B01001_008E,Estimate!!Total:!!Male:!!20 years,Sex by Age,int,B01001
46,B01001_009E,Estimate!!Total:!!Male:!!21 years,Sex by Age,int,B01001
48,B01001_010E,Estimate!!Total:!!Male:!!22 to 24 years,Sex by Age,int,B01001


## Reclaim new naming format

Follow Eli's comment on the PR:

`ok, now that i've looked at ont of the 2022 tables in the geodatabase, the reason you're getting no results is the naming convention has changed. Your PR includes an update for the geoid column, but there are other systematic changes. In the new tables, the variables are named (as an example): B02001_E001. We need to have processing that anticipates this format, then converts it to the canonical form (like the json tables, B02001_001E (where E/M is the final character of the variable rather than the leading character)`

In [19]:
import pandas as pd
import geopandas as gpd

import re
from pathlib import Path
import pyarrow.parquet as pq

from IPython.display import display

In [20]:
# Adjust this path if needed
BUILD_ROOT = Path("../build")

DIR_2021 = BUILD_ROOT / "2021_bg"
DIR_2022 = BUILD_ROOT / "2022_bg"
REPORT_DIR = BUILD_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("2021 dir:", DIR_2021.resolve())
print("2022 dir:", DIR_2022.resolve())
print("Report dir:", REPORT_DIR.resolve())

2021 dir: /home/dylan/projects/geosnap/build/2021_bg
2022 dir: /home/dylan/projects/geosnap/build/2022_bg
Report dir: /home/dylan/projects/geosnap/build/reports


## Inspect the new naming on one file

In [23]:
test1 = pd.read_parquet(f'{DIR_2022}/acs_2022_X14_SCHOOL_ENROLLMENT_bg.parquet')

In [24]:
test1.head()

,B14002_E001,B14002_E002,B14002_E003,B14002_E004,B14002_E005,B14002_E006,B14002_E007,B14002_E008,B14002_E009,B14002_E010,...,B14007I_E010,B14007I_E011,B14007I_E012,B14007I_E013,B14007I_E014,B14007I_E015,B14007I_E016,B14007I_E017,B14007I_E018,B14007I_E019
GEOIDFQ,,,,,,,,,,,,,,,,,,,,,
1500000US010179548002,1375.0,529.0,45.0,24.0,24.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,21.0
1500000US010179548004,773.0,409.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,27.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0
1500000US010179548003,281.0,85.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
1500000US010150011031,539.0,321.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1500000US010150024003,970.0,421.0,93.0,0.0,0.0,0.0,0.0,0.0,0.0,33.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
test1_2021 = pd.read_parquet(f'{DIR_2021}/acs_2021_X14_SCHOOL_ENROLLMENT_bg.parquet')

In [29]:
test1_2021.head()

,B14002_001E,B14002_002E,B14002_003E,B14002_004E,B14002_005E,B14002_006E,B14002_007E,B14002_008E,B14002_009E,B14002_010E,...,B14007I_010E,B14007I_011E,B14007I_012E,B14007I_013E,B14007I_014E,B14007I_015E,B14007I_016E,B14007I_017E,B14007I_018E,B14007I_019E
GEOID,,,,,,,,,,,,,,,,,,,,,
010010201001,691.0,296.0,46.0,4.0,0.0,4.0,0.0,0.0,0.0,31.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0
010010201002,1038.0,558.0,145.0,0.0,0.0,0.0,0.0,0.0,0.0,87.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0
010010202001,782.0,324.0,77.0,7.0,0.0,7.0,0.0,0.0,0.0,29.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
010010202002,1146.0,703.0,67.0,0.0,0.0,0.0,0.0,0.0,0.0,28.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
010010203001,2667.0,1256.0,329.0,0.0,0.0,0.0,25.0,25.0,0.0,117.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,5.0


In [25]:
test2 = pd.read_parquet(f'{DIR_2022}/acs_2022_X03_HISPANIC_OR_LATINO_ORIGIN_bg.parquet')

In [26]:
test2.head()

,B03002_E001,B03002_E002,B03002_E003,B03002_E004,B03002_E005,B03002_E006,B03002_E007,B03002_E008,B03002_E009,B03002_E010,...,B03002_E015,B03002_E016,B03002_E017,B03002_E018,B03002_E019,B03002_E020,B03002_E021,B03003_E001,B03003_E002,B03003_E003
GEOIDFQ,,,,,,,,,,,,,,,,,,,,,
1500000US010179548002,1375.0,1340.0,149.0,1191.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1375.0,1340.0,35.0
1500000US010179548004,797.0,767.0,450.0,314.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,30.0,0.0,0.0,0.0,797.0,767.0,30.0
1500000US010179548003,281.0,276.0,138.0,138.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,5.0,0.0,0.0,0.0,281.0,276.0,5.0
1500000US010150011031,560.0,560.0,560.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,560.0,560.0,0.0
1500000US010150024003,1003.0,1003.0,871.0,45.0,0.0,0.0,0.0,0.0,87.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1003.0,1003.0,0.0


In [30]:
test2_2021 = pd.read_parquet(f'{DIR_2021}/acs_2021_X03_HISPANIC_OR_LATINO_ORIGIN_bg.parquet')

In [31]:
test2_2021.head()

,B03002_001E,B03002_002E,B03002_003E,B03002_004E,B03002_005E,B03002_006E,B03002_007E,B03002_008E,B03002_009E,B03002_010E,...,B03002_015E,B03002_016E,B03002_017E,B03002_018E,B03002_019E,B03002_020E,B03002_021E,B03003_001E,B03003_002E,B03003_003E
GEOID,,,,,,,,,,,,,,,,,,,,,
010010201001,693.0,674.0,587.0,16.0,0.0,0.0,0.0,0.0,71.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,693.0,674.0,19.0
010010201002,1098.0,1089.0,887.0,155.0,0.0,38.0,0.0,0.0,9.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1098.0,1089.0,9.0
010010202001,844.0,834.0,336.0,421.0,0.0,0.0,0.0,0.0,77.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,844.0,834.0,10.0
010010202002,1166.0,1166.0,439.0,667.0,0.0,0.0,0.0,8.0,52.0,27.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1166.0,1166.0,0.0
010010203001,2685.0,2672.0,2011.0,531.0,0.0,26.0,0.0,0.0,104.0,0.0,...,0.0,0.0,0.0,7.0,6.0,6.0,0.0,2685.0,2672.0,13.0


It would be really cool if the 'E' moving was the only naming convention change with the new vintage

In [32]:
# New 2022-style ACS naming:
#   B02001_E001
#   B02001_M001
NEW_STYLE_ACS_RE = re.compile(r"^([A-Z0-9]+)_([EM])(\d{3})$", re.IGNORECASE)

# Canonical ACS naming:
#   B02001_001E
#   B02001_001M
CANONICAL_ACS_RE = re.compile(r"^([A-Z0-9]+)_(\d{3})([EM])$", re.IGNORECASE)

# Flexible GEOID-like matcher
GEOID_RE = re.compile(r"^GEOID([A-Z_].*)?$", re.IGNORECASE)


def read_parquet_columns(path: Path) -> list[str]:
    """Read parquet schema only, not the data."""
    schema = pq.ParquetFile(path).schema_arrow
    return schema.names


def canonicalize_column(col: str) -> str:
    """
    Normalize ACS variable names to canonical form.

    Examples:
      B02001_E001 -> B02001_001E
      B02001_M001 -> B02001_001M
      B02001_001E -> B02001_001E
    """
    c = col.strip()

    m = NEW_STYLE_ACS_RE.match(c)
    if m:
        stem, suffix, digits = m.groups()
        return f"{stem.upper()}_{digits}{suffix.upper()}"

    m = CANONICAL_ACS_RE.match(c)
    if m:
        stem, digits, suffix = m.groups()
        return f"{stem.upper()}_{digits}{suffix.upper()}"

    return c


def classify_column(col: str) -> str:
    c = col.strip()

    if GEOID_RE.match(c) or c in {"GEOIDFQ", "GEOID_Data"}:
        return "geoid_like"

    if NEW_STYLE_ACS_RE.match(c):
        return "acs_new_style"

    if CANONICAL_ACS_RE.match(c):
        return "acs_canonical"

    return "other"


def layer_key(path: Path) -> str:
    """
    Convert a filename into a year-agnostic layer key.
    """
    name = path.name

    if re.fullmatch(r"acs_\d{4}_bg\.parquet", name):
        return "ALL_BG"

    if re.fullmatch(r"acs_demographic_profile_\d{4}_bg\.parquet", name):
        return "DEMOGRAPHIC_PROFILE"

    m = re.fullmatch(r"acs_\d{4}_(.+?)_bg\.parquet", name)
    if m:
        return m.group(1)

    return name

## Sanity check: do the helper functions work?

Expected: 
- B02001_E001 should become B02001_001E
- B02001_M001 should become B02001_001M
- canonical names should stay unchanged
- GEOID-like columns should stay unchanged

In [33]:
test_cols = [
    "B02001_E001",
    "B02001_M001",
    "B02001_001E",
    "B19013_001E",
    "GEOID",
    "GEOIDFQ",
    "GEOID_Data",
    "NAME",
]

test_df = pd.DataFrame({
    "original": test_cols,
    "classification": [classify_column(c) for c in test_cols],
    "canonical": [canonicalize_column(c) for c in test_cols],
})

display(test_df)

,original,classification,canonical
0,B02001_E001,acs_new_style,B02001_001E
1,B02001_M001,acs_new_style,B02001_001M
2,B02001_001E,acs_canonical,B02001_001E
3,B19013_001E,acs_canonical,B19013_001E
4,GEOID,geoid_like,GEOID
5,GEOIDFQ,geoid_like,GEOIDFQ
6,GEOID_Data,geoid_like,GEOID_Data
7,NAME,other,NAME


## Compare the files between vintages

Just verifying comparable tables and identifying what is new

In [43]:
def parse_parquet_filename(path: Path) -> dict:
    """
    Parse known ACS parquet filenames into structured parts and normalize
    year-specific pieces so 2021 and 2022 comparable files align.
    """
    name = path.name

    # acs_demographic_profile_2022_bg.parquet
    m = re.match(
        r"^acs_demographic_profile_(\d{4})_(\w+)\.parquet$",
        name,
        flags=re.IGNORECASE,
    )
    if m:
        year, geography = m.groups()
        return {
            "file": name,
            "year": int(year),
            "kind": "demographic_profile",
            "x_code": None,
            "table_name": "demographic_profile",
            "geography": geography,
            "group_key": f"demographic_profile::{geography}",
            "sort_key": (9998, "demographic_profile", geography),
        }

    # acs_2022_X29_VOTING_AGE_POPULATION_bg.parquet
    m = re.match(
        r"^acs_(\d{4})_(X\d{2})_(.+)_(\w+)\.parquet$",
        name,
        flags=re.IGNORECASE,
    )
    if m:
        year, x_code, table_name, geography = m.groups()
        x_code = x_code.upper()
        return {
            "file": name,
            "year": int(year),
            "kind": "x_table",
            "x_code": x_code,
            "table_name": table_name,
            "geography": geography,
            "group_key": f"{x_code}::{table_name}::{geography}",
            "sort_key": (int(x_code[1:]), table_name, geography),
        }

    # acs_2022_ACS_2022_5YR_BG_bg.parquet
    # normalize ACS_2021_5YR_BG and ACS_2022_5YR_BG to ACS_5YR_BG
    m = re.match(
        r"^acs_(\d{4})_(ACS_\d{4}_5YR_[A-Z]+)_(\w+)\.parquet$",
        name,
        flags=re.IGNORECASE,
    )
    if m:
        year, source_name, geography = m.groups()
        source_name_norm = re.sub(r"ACS_\d{4}_5YR_", "ACS_5YR_", source_name, flags=re.IGNORECASE)
        return {
            "file": name,
            "year": int(year),
            "kind": "whole_gdb",
            "x_code": None,
            "table_name": source_name_norm,
            "geography": geography,
            "group_key": f"whole_gdb::{source_name_norm}::{geography}",
            "sort_key": (9996, source_name_norm, geography),
        }

    # acs_2022_bg.parquet
    m = re.match(
        r"^acs_(\d{4})_(\w+)\.parquet$",
        name,
        flags=re.IGNORECASE,
    )
    if m:
        year, geography = m.groups()
        return {
            "file": name,
            "year": int(year),
            "kind": "combined",
            "x_code": None,
            "table_name": "combined",
            "geography": geography,
            "group_key": f"combined::{geography}",
            "sort_key": (9997, "combined", geography),
        }

    return {
        "file": name,
        "year": None,
        "kind": "unknown",
        "x_code": None,
        "table_name": name,
        "geography": None,
        "group_key": f"unknown::{name}",
        "sort_key": (9999, name, ""),
    }

In [44]:
files_2021 = sorted(DIR_2021.glob("*.parquet"))
files_2022 = sorted(DIR_2022.glob("*.parquet"))

parsed_2021 = pd.DataFrame([parse_parquet_filename(p) for p in files_2021])
parsed_2022 = pd.DataFrame([parse_parquet_filename(p) for p in files_2022])

print(f"2021 parquet files: {len(parsed_2021)}")
print(f"2022 parquet files: {len(parsed_2022)}")

2021 parquet files: 26
2022 parquet files: 34


In [45]:
compare_files_df = (
    parsed_2021.rename(columns={"file": "file_2021", "year": "year_2021"})
    .merge(
        parsed_2022.rename(columns={"file": "file_2022", "year": "year_2022"}),
        on=["group_key", "kind", "x_code", "table_name", "geography", "sort_key"],
        how="outer",
    )
    .sort_values(["sort_key", "kind", "table_name", "group_key"])
    .reset_index(drop=True)
)

compare_files_df["exists_2021"] = compare_files_df["file_2021"].notna()
compare_files_df["exists_2022"] = compare_files_df["file_2022"].notna()

display(
    compare_files_df[
        [
            "file_2021",
            "file_2022",
            "exists_2021",
            "exists_2022",
        ]
    ]
)

,file_2021,file_2022,exists_2021,exists_2022
0,acs_2021_X01_AGE_AND_SEX_bg.parquet,acs_2022_X01_AGE_AND_SEX_bg.parquet,True,True
1,acs_2021_X02_RACE_bg.parquet,acs_2022_X02_RACE_bg.parquet,True,True
2,acs_2021_X03_HISPANIC_OR_LATINO_ORIGIN_bg.parquet,acs_2022_X03_HISPANIC_OR_LATINO_ORIGIN_bg.parquet,True,True
3,NaN,acs_2022_X04_ANCESTRY_bg.parquet,False,True
4,NaN,acs_2022_X05_FOREIGN_BORN_CITIZENSHIP_bg.parquet,False,True
5,NaN,acs_2022_X06_PLACE_OF_BIRTH_bg.parquet,False,True
6,acs_2021_X07_MIGRATION_bg.parquet,acs_2022_X07_MIGRATION_bg.parquet,True,True
7,acs_2021_X08_COMMUTING_bg.parquet,acs_2022_X08_COMMUTING_bg.parquet,True,True
8,acs_2021_X09_CHILDREN_HOUSEHOLD_RELATIONSHIP_b...,acs_2022_X09_CHILDREN_HOUSEHOLD_RELATIONSHIP_b...,True,True
9,NaN,acs_2022_X10_GRANDPARENTS_GRANDCHILDREN_bg.par...,False,True


This cell 

In [50]:
inspect21 = pd.DataFrame({
    "column": cols21,
    "classification": [classify_column(c) for c in cols21],
    "canonical": [canonicalize_column(c) for c in cols21],
    "changed": [c != canonicalize_column(c) for c in cols21],
})
inspect22 = pd.DataFrame({
    "column": cols22,
    "classification": [classify_column(c) for c in cols22],
    "canonical": [canonicalize_column(c) for c in cols22],
    "changed": [c != canonicalize_column(c) for c in cols22],
})

print("2021 classification counts")
display(inspect21["classification"].value_counts().rename_axis("classification").reset_index(name="count"))

print("2022 classification counts")
display(inspect22["classification"].value_counts().rename_axis("classification").reset_index(name="count"))

2021 classification counts


,classification,count
0,acs_canonical,35
1,geoid_like,1


2022 classification counts


,classification,count
0,acs_new_style,37
1,geoid_like,1


We want to see no columns changed for 2021, but many for 2022

In [51]:
print("2021 columns changed by canonicalization")
display(inspect21[inspect21["changed"]].head(30))

print("2022 columns changed by canonicalization")
display(inspect22[inspect22["changed"]].head(30))

2021 columns changed by canonicalization


,column,classification,canonical,changed


2022 columns changed by canonicalization


,column,classification,canonical,changed
0,B02001_E001,acs_new_style,B02001_001E,True
1,B02001_E002,acs_new_style,B02001_002E,True
2,B02001_E003,acs_new_style,B02001_003E,True
3,B02001_E004,acs_new_style,B02001_004E,True
4,B02001_E005,acs_new_style,B02001_005E,True
5,B02001_E006,acs_new_style,B02001_006E,True
6,B02001_E007,acs_new_style,B02001_007E,True
7,B02001_E008,acs_new_style,B02001_008E,True
8,B02001_E009,acs_new_style,B02001_009E,True
9,B02001_E010,acs_new_style,B02001_010E,True


In [52]:
raw_overlap = len(set(cols21) & set(cols22))
canon_overlap = len({canonicalize_column(c) for c in cols21} & {canonicalize_column(c) for c in cols22})

comparison_df = pd.DataFrame([{
    "layer": layer,
    "n_cols_2021_raw": len(cols21),
    "n_cols_2022_raw": len(cols22),
    "raw_overlap": raw_overlap,
    "canonical_overlap": canon_overlap,
    "raw_only_2021": len(set(cols21) - set(cols22)),
    "raw_only_2022": len(set(cols22) - set(cols21)),
    "canonical_only_2021": len({canonicalize_column(c) for c in cols21} - {canonicalize_column(c) for c in cols22}),
    "canonical_only_2022": len({canonicalize_column(c) for c in cols22} - {canonicalize_column(c) for c in cols21}),
}])

display(comparison_df)

,layer,n_cols_2021_raw,n_cols_2022_raw,raw_overlap,canonical_overlap,raw_only_2021,raw_only_2022,canonical_only_2021,canonical_only_2022
0,X02_RACE,36,38,0,35,36,38,1,3


The variables specifically:

In [54]:
canon21 = pd.DataFrame({
    "canonical": [canonicalize_column(c) for c in cols21],
    "raw_2021": cols21,
}).drop_duplicates()

canon22 = pd.DataFrame({
    "canonical": [canonicalize_column(c) for c in cols22],
    "raw_2022": cols22,
}).drop_duplicates()

aligned = canon21.merge(canon22, on="canonical", how="outer")
aligned["different_raw_names"] = (
    aligned["raw_2021"].notna() &
    aligned["raw_2022"].notna() &
    (aligned["raw_2021"] != aligned["raw_2022"])
)

display(aligned[aligned["different_raw_names"]])

,canonical,raw_2021,raw_2022,different_raw_names
0,B02001_001E,B02001_001E,B02001_E001,True
1,B02001_002E,B02001_002E,B02001_E002,True
2,B02001_003E,B02001_003E,B02001_E003,True
3,B02001_004E,B02001_004E,B02001_E004,True
4,B02001_005E,B02001_005E,B02001_E005,True
5,B02001_006E,B02001_006E,B02001_E006,True
6,B02001_007E,B02001_007E,B02001_E007,True
7,B02001_008E,B02001_008E,B02001_E008,True
8,B02001_009E,B02001_009E,B02001_E009,True
9,B02001_010E,B02001_010E,B02001_E010,True


The mismatch is almost entirely due to naming convention changes, not missing data.

the canonicalization rule recovers ~97% (35/36) of variables.

The remaining differences are small and real (1 dropped, 3 added), not a pipeline failure.

## New variables in 2022

The two cells below show the unaligned variables that are not covered by the new rule.

In [56]:
display(aligned[aligned["raw_2022"].isna()])

,canonical,raw_2021,raw_2022,different_raw_names
37,GEOID,GEOID,NaN,False


The GEOID difference we know about, but the two variables below are new to this table this vintage.

In [57]:
display(aligned[aligned["raw_2021"].isna()])

,canonical,raw_2021,raw_2022,different_raw_names
35,C02003_020E,NaN,C02003_E020,False
36,C02003_021E,NaN,C02003_E021,False
38,GEOIDFQ,NaN,GEOIDFQ,False


## GEOID grepper

hoping that `classify_columns` will recognize multiple GEOID-like columns

In [58]:
geoid21 = [c for c in cols21 if classify_column(c) == "geoid_like"]
geoid22 = [c for c in cols22 if classify_column(c) == "geoid_like"]

print("2021 GEOID-like columns:", geoid21)
print("2022 GEOID-like columns:", geoid22)

2021 GEOID-like columns: ['GEOID']
2022 GEOID-like columns: ['GEOIDFQ']


# Inspect all layers

Now we put these new functions onto all the files

In [59]:
def inspect_file(path: Path, year: int) -> pd.DataFrame:
    cols = read_parquet_columns(path)
    return pd.DataFrame({
        "year": year,
        "file": path.name,
        "layer": layer_key(path),
        "column": cols,
        "classification": [classify_column(c) for c in cols],
        "canonical": [canonicalize_column(c) for c in cols],
        "changed": [c != canonicalize_column(c) for c in cols],
    })


all_column_frames = []

for layer in all_layers:
    if layer in idx_2021:
        all_column_frames.append(inspect_file(idx_2021[layer], 2021))
    if layer in idx_2022:
        all_column_frames.append(inspect_file(idx_2022[layer], 2022))

all_columns_df = pd.concat(all_column_frames, ignore_index=True)

display(all_columns_df.head()) # can look closer here
print("Total inspected columns:", len(all_columns_df))

,year,file,layer,column,classification,canonical,changed
0,2021,acs_2021_ACS_2021_5YR_BG_bg.parquet,ACS_2021_5YR_BG,STATEFP,other,STATEFP,False
1,2021,acs_2021_ACS_2021_5YR_BG_bg.parquet,ACS_2021_5YR_BG,COUNTYFP,other,COUNTYFP,False
2,2021,acs_2021_ACS_2021_5YR_BG_bg.parquet,ACS_2021_5YR_BG,TRACTCE,other,TRACTCE,False
3,2021,acs_2021_ACS_2021_5YR_BG_bg.parquet,ACS_2021_5YR_BG,BLKGRPCE,other,BLKGRPCE,False
4,2021,acs_2021_ACS_2021_5YR_BG_bg.parquet,ACS_2021_5YR_BG,NAMELSAD,other,NAMELSAD,False


Total inspected columns: 16998


## Summarize across all layers

In [60]:
file_summary_df = (
    all_columns_df
    .groupby(["year", "file", "layer", "classification"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

if "acs_canonical" not in file_summary_df.columns:
    file_summary_df["acs_canonical"] = 0
if "acs_new_style" not in file_summary_df.columns:
    file_summary_df["acs_new_style"] = 0
if "geoid_like" not in file_summary_df.columns:
    file_summary_df["geoid_like"] = 0
if "other" not in file_summary_df.columns:
    file_summary_df["other"] = 0

changed_summary = (
    all_columns_df
    .groupby(["year", "file", "layer"])["changed"]
    .sum()
    .reset_index(name="n_changed_by_canonicalization")
)

file_summary_df = file_summary_df.merge(
    changed_summary,
    on=["year", "file", "layer"],
    how="left"
)

display(file_summary_df.sort_values(["layer", "year"]))

,year,file,layer,acs_canonical,acs_new_style,geoid_like,other,n_changed_by_canonicalization
0,2021,acs_2021_ACS_2021_5YR_BG_bg.parquet,ACS_2021_5YR_BG,0,0,2,14,0
26,2022,acs_2022_ACS_2022_5YR_BG_bg.parquet,ACS_2022_5YR_BG,0,0,2,14,0
24,2021,acs_2021_bg.parquet,ALL_BG,0,0,1,37,0
58,2022,acs_2022_bg.parquet,ALL_BG,0,0,1,1,0
25,2021,acs_demographic_profile_2021_bg.parquet,DEMOGRAPHIC_PROFILE,4159,0,2,14,0
59,2022,acs_demographic_profile_2022_bg.parquet,DEMOGRAPHIC_PROFILE,0,4261,1,15,4261
1,2021,acs_2021_X01_AGE_AND_SEX_bg.parquet,X01_AGE_AND_SEX,80,0,1,0,0
27,2022,acs_2022_X01_AGE_AND_SEX_bg.parquet,X01_AGE_AND_SEX,0,80,1,0,80
2,2021,acs_2021_X02_RACE_bg.parquet,X02_RACE,35,0,1,0,0
28,2022,acs_2022_X02_RACE_bg.parquet,X02_RACE,0,37,1,0,37


In [63]:
def layer_compare(layer: str) -> dict:
    cols21 = set(read_parquet_columns(idx_2021[layer])) if layer in idx_2021 else set()
    cols22 = set(read_parquet_columns(idx_2022[layer])) if layer in idx_2022 else set()

    canon21 = {canonicalize_column(c) for c in cols21}
    canon22 = {canonicalize_column(c) for c in cols22}

    return {
        "layer": layer,
        "exists_2021": layer in idx_2021,
        "exists_2022": layer in idx_2022,
        "n_cols_2021_raw": len(cols21),
        "n_cols_2022_raw": len(cols22),
        "raw_overlap": len(cols21 & cols22),
        "canonical_overlap": len(canon21 & canon22),
        "raw_only_2021": len(cols21 - cols22),
        "raw_only_2022": len(cols22 - cols21),
        "canonical_only_2021": len(canon21 - canon22),
        "canonical_only_2022": len(canon22 - canon21),
    }

layer_comparison_df = pd.DataFrame([layer_compare(layer) for layer in all_layers])
display(layer_comparison_df.sort_values("layer"))


# Ignore the first two rows. In the last two columns (canonical_only), the columns which have `1` are going to be the differently named GEOID columns. 
# Some tables have large differences: Demographic Profile, Housing Characteristics, and Race all have > 1
# Some tables are entirely new in 2022 (X04, X05, X06, X10, X13, X18, X26, X98)

,layer,exists_2021,exists_2022,n_cols_2021_raw,n_cols_2022_raw,raw_overlap,canonical_overlap,raw_only_2021,raw_only_2022,canonical_only_2021,canonical_only_2022
0,ACS_2021_5YR_BG,True,False,16,0,0,0,16,0,16,0
1,ACS_2022_5YR_BG,False,True,0,16,0,0,0,16,0,16
2,ALL_BG,True,True,38,2,2,2,36,0,36,0
3,DEMOGRAPHIC_PROFILE,True,True,4175,4277,14,4173,4161,4263,2,104
4,X01_AGE_AND_SEX,True,True,81,81,0,80,81,81,1,1
5,X02_RACE,True,True,36,38,0,35,36,38,1,3
6,X03_HISPANIC_OR_LATINO_ORIGIN,True,True,25,25,0,24,25,25,1,1
7,X04_ANCESTRY,False,True,0,1,0,0,0,1,0,1
8,X05_FOREIGN_BORN_CITIZENSHIP,False,True,0,1,0,0,0,1,0,1
9,X06_PLACE_OF_BIRTH,False,True,0,1,0,0,0,1,0,1


In [65]:
canon_pairs = (
    all_columns_df[["year", "layer", "column", "canonical"]]
    .drop_duplicates()
)

canon_pivot = canon_pairs.pivot_table(
    index=["layer", "canonical"],
    columns="year",
    values="column",
    aggfunc="first"
).reset_index()

canon_pivot.columns.name = None
if 2021 not in canon_pivot.columns:
    canon_pivot[2021] = None
if 2022 not in canon_pivot.columns:
    canon_pivot[2022] = None

canon_pivot = canon_pivot.rename(columns={2021: "raw_2021", 2022: "raw_2022"})
canon_pivot["different_raw_names"] = (
    canon_pivot["raw_2021"].notna() &
    canon_pivot["raw_2022"].notna() &
    (canon_pivot["raw_2021"] != canon_pivot["raw_2022"])
)

renamed_df = canon_pivot[canon_pivot["different_raw_names"]].sort_values(["layer", "canonical"])

display(renamed_df)
print("Rows with same canonical variable but different raw names:", len(renamed_df))

,layer,canonical,raw_2021,raw_2022,different_raw_names
72,DEMOGRAPHIC_PROFILE,B01001_001E,B01001_001E,B01001_E001,True
73,DEMOGRAPHIC_PROFILE,B01001_002E,B01001_002E,B01001_E002,True
74,DEMOGRAPHIC_PROFILE,B01001_003E,B01001_003E,B01001_E003,True
75,DEMOGRAPHIC_PROFILE,B01001_004E,B01001_004E,B01001_E004,True
76,DEMOGRAPHIC_PROFILE,B01001_005E,B01001_005E,B01001_E005,True
...,...,...,...,...,...
8657,X99_IMPUTATION,B99283_001E,B99283_001E,B99283_E001,True
8658,X99_IMPUTATION,B99283_002E,B99283_002E,B99283_E002,True
8659,X99_IMPUTATION,B99283_003E,B99283_003E,B99283_E003,True
8660,X99_IMPUTATION,B99283_004E,B99283_004E,B99283_E004,True


Rows with same canonical variable but different raw names: 8318


# Housing characteristics??

Why is this table so much different? Are these really 100 new variables?

In [70]:
BUILD_ROOT = Path("../build")

file_2021 = BUILD_ROOT / "2021_bg" / "acs_2021_X25_HOUSING_CHARACTERISTICS_bg.parquet"
file_2022 = BUILD_ROOT / "2022_bg" / "acs_2022_X25_HOUSING_CHARACTERISTICS_bg.parquet"

print("2021 exists:", file_2021.exists(), file_2021)
print("2022 exists:", file_2022.exists(), file_2022)

2021 exists: True ../build/2021_bg/acs_2021_X25_HOUSING_CHARACTERISTICS_bg.parquet
2022 exists: True ../build/2022_bg/acs_2022_X25_HOUSING_CHARACTERISTICS_bg.parquet


In [71]:
def read_parquet_columns(path: Path) -> list[str]:
    return pq.ParquetFile(path).schema_arrow.names

cols_2021 = read_parquet_columns(file_2021)
cols_2022 = read_parquet_columns(file_2022)

print("n_cols_2021_raw:", len(cols_2021))
print("n_cols_2022_raw:", len(cols_2022))

n_cols_2021_raw: 871
n_cols_2022_raw: 971


Canonicalize the columns

In [73]:
df21 = pd.DataFrame({"raw_2021": cols_2021})
df21["canonical"] = df21["raw_2021"].map(canonicalize_column)

df22 = pd.DataFrame({"raw_2022": cols_2022})
df22["canonical"] = df22["raw_2022"].map(canonicalize_column)

In [75]:
aligned = (
    df21.merge(df22, on="canonical", how="outer")
    .sort_values("canonical")
    .reset_index(drop=True)
)

aligned["present_2021"] = aligned["raw_2021"].notna()
aligned["present_2022"] = aligned["raw_2022"].notna()

display(aligned.head(5))

,raw_2021,canonical,raw_2022,present_2021,present_2022
0,B25001_001E,B25001_001E,B25001_E001,True,True
1,B25002_001E,B25002_001E,B25002_E001,True,True
2,B25002_002E,B25002_002E,B25002_E002,True,True
3,B25002_003E,B25002_003E,B25002_E003,True,True
4,B25003A_001E,B25003A_001E,B25003A_E001,True,True


## Now get vars only present in 2022

In [83]:
only_2021 = aligned[
    aligned["raw_2021"].notna() & aligned["raw_2022"].isna()
].copy()

acs_pattern = r"^[A-Z0-9]+_\d{3}[EM]$"

acs_only_2022 = only_2022[
    only_2022["canonical"].str.match(acs_pattern, na=False)
].copy()

print("ACS variables only in 2022:", len(acs_only_2022))
display(acs_only_2022.head())

ACS variables only in 2022: 100


,raw_2021,canonical,raw_2022,present_2021,present_2022
73,NaN,B25008A_001E,B25008A_E001,False,True
74,NaN,B25008A_002E,B25008A_E002,False,True
75,NaN,B25008A_003E,B25008A_E003,False,True
76,NaN,B25008B_001E,B25008B_E001,False,True
77,NaN,B25008B_002E,B25008B_E002,False,True


In [81]:
acs_only_2022["prefix"] = acs_only_2022["canonical"].str.extract(r"^([A-Z0-9]+)_")

prefix_counts = (
    acs_only_2022["prefix"]
    .value_counts()
    .rename_axis("table_prefix")
    .reset_index(name="n_new_vars")
)

display(prefix_counts)

,table_prefix,n_new_vars
0,B25136,13
1,B25140,13
2,B25137,9
3,B25008A,3
4,B25008E,3
5,B25008B,3
6,B25008C,3
7,B25008D,3
8,B25008I,3
9,B25010A,3
